In [7]:
import os

os.chdir("../")

In [8]:
from llm import ask_groq
from IPython.display import Markdown, display

# Example usage (teaching-friendly)
question = "Who won the WB election?"
answer = ask_groq(question)

# Nicely formatted output for notebook display
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Who won the WB election?

**Answer:** I'm not aware of any specific information about a "WB election." Could you please provide more context or clarify which election you are referring to? I'll do my best to provide you with the most up-to-date information.

In [9]:
wikipedia_context = """
Legislative Assembly elections were held in West Bengal to elect all 294 members of the West Bengal Legislative Assembly in two phases on 23 and 29 April 2026,[4] with the votes counted and results for 293 seats released on 4 May 2026.

Over 9 million voters were removed through the Special Intensive Revision (SIR) prior to the elections. This move has been criticized as erosion of democracy in India.[5][6]

The election saw a historic defeat for the incumbent All India Trinamool Congress which was ruling since 2011, with the Bharatiya Janata Party becoming the first right-wing party to be elected in the state since assembly elections first began in 1937. With a voter turnout of 92.93%, the election was the most widely participated in West Bengal, surpassing the 2011 election.

In a move unprecedented in Indian politics, Mamata Banerjee, the incumbent Trinamool Congress chief minister, refused to resign her office despite losing her seat and a majority in the assembly as a result of the election, alleging irregularities in its conduct.[7][8][9] Her tenure as chief minister came to an end after the dissolution of the assembly by the governor at the end of its term on 7 May 2026.
"""

question = "Who won the WB election?"
question_with_context = f"{wikipedia_context}\n\nQuestion: {question}"
answer = ask_groq(question_with_context)

# Nicely formatted output for notebook display
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Who won the WB election?

**Answer:** The Bharatiya Janata Party (BJP) won the West Bengal Legislative Assembly election, becoming the first right-wing party to be elected in the state since assembly elections first began in 1937, and ending the 11-year rule of the All India Trinamool Congress (AITC).

In [10]:
from pydantic import BaseModel, Field
from typing import Any

class ToolUsage(BaseModel):
    tool: str = Field(description="Exact name of the tool to use")
    args: dict[str, Any] = Field(description="Arguments to pass to the tool")
    reason: str = Field(description="Reason for using the tool")


schema = ToolUsage.model_json_schema()
schema


{'properties': {'tool': {'description': 'Exact name of the tool to use',
   'title': 'Tool',
   'type': 'string'},
  'args': {'additionalProperties': True,
   'description': 'Arguments to pass to the tool',
   'title': 'Args',
   'type': 'object'},
  'reason': {'description': 'Reason for using the tool',
   'title': 'Reason',
   'type': 'string'}},
 'required': ['tool', 'args', 'reason'],
 'title': 'ToolUsage',
 'type': 'object'}

In [11]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# MCP server endpoint — override via the MARKETINTEL_ENDPOINT environment variable.
SERVER_ENDPOINT = "http://127.0.0.1:8000/mcp"
transport = StreamableHttpTransport(url=SERVER_ENDPOINT)
client = Client(transport)

In [12]:
async with client:
    tools = await client.list_tools()
    prompt = f"What tools would you use to answer the following question: `{question}`? Here is the list of available tools: `{tools}`. Your answer should be a JSON array of `{schema}` objects, where each object specifies a tool to use, the arguments to pass to that tool, and the reason for using it."
    response = ask_groq(question=prompt)
    print(response)

```json
[
    {
        "tool": "web_search",
        "args": {
            "query": "West Bengal election results"
        },
        "reason": "To find the latest information on the West Bengal election results"
    }
]
```


In [13]:
import re
import json

def get_json(response: str) -> list[ToolUsage]:
    json_str = re.search(r"\[.*\]", response, re.DOTALL).group(0)
    return json.loads(json_str)

tool_usages = get_json(response)
tool_usages

[{'tool': 'web_search',
  'args': {'query': 'West Bengal election results'},
  'reason': 'To find the latest information on the West Bengal election results'}]

In [14]:
async with client:
    for tool_usage in tool_usages:
        print(f"Calling tool: {tool_usage['tool']} with arguments: {tool_usage['args']} because {tool_usage['reason']}")
        tool_response = await client.call_tool(tool_usage["tool"], arguments=tool_usage["args"])
        print(f"Response from tool {tool_usage['tool']}: \n{tool_response.content[0].text}")

Calling tool: web_search with arguments: {'query': 'West Bengal election results'} because To find the latest information on the West Bengal election results
Response from tool web_search: 
{'query': 'West Bengal election results', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.youtube.com/watch?v=Qn1A9KxraMo', 'title': "West Bengal Election Results LIVE: Mamata's First ... - YouTube", 'content': 'West Bengal assembly elections | Counting of votes underway for 293 Assembly constituencies in West Bengal. The state is set to witness a', 'score': 0.999617, 'raw_content': None}, {'url': 'https://en.wikipedia.org/wiki/Elections_in_West_Bengal', 'title': 'Elections in West Bengal - Wikipedia', 'content': '| 1971 | 5th |  | **CPI(M) "Communist Party of India (Marxist)") 20** |  | INC 13 |  | **CPI 3** |  | **RSP "Revolutionary Socialist Party (India)") 1** | BAC 1, PSP 1, Ind "Independent (politician)") 1 |. | 1977 | 6th |  | **CPI(M) "Communist Pa

In [15]:
import ast

tool_response_data = ast.literal_eval(tool_response.content[0].text)
tool_response_data

{'query': 'West Bengal election results',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=Qn1A9KxraMo',
   'title': "West Bengal Election Results LIVE: Mamata's First ... - YouTube",
   'content': 'West Bengal assembly elections | Counting of votes underway for 293 Assembly constituencies in West Bengal. The state is set to witness a',
   'score': 0.999617,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Elections_in_West_Bengal',
   'title': 'Elections in West Bengal - Wikipedia',
   'content': '| 1971 | 5th |  | **CPI(M) "Communist Party of India (Marxist)") 20** |  | INC 13 |  | **CPI 3** |  | **RSP "Revolutionary Socialist Party (India)") 1** | BAC 1, PSP 1, Ind "Independent (politician)") 1 |. | 1977 | 6th |  | **CPI(M) "Communist Party of India (Marxist)") 17** |  | **BLD 15** |  | INC 3 |  | **RSP "Revolutionary Socialist Party (India)") 3** | AIFB 3, Ind "Independent (politician)") 1 | 42 |. |

In [16]:
formatting_instructions = """Markdown formatting instructions: 1. Use headings (##) for each section. 2. Use bullet points for lists. 3. Include URLs as hyperlinks. 4. Use bold for important points. 5. Keep the response concise and informative. 6. Use tables if needed to present structured data clearly."""
query = question
search_results = tool_response_data

async with client:
    prompt = f"""You are a helpful assistant who takes the user query and search results and provides a concise and informative response. Use the following formatting guidelines: ```{formatting_instructions}```. Here is the user query: `{query}`. Here are the search results: ```{search_results}```. Please provide a well-formatted response based on the query and search results."""
    analysis_response = ask_groq(question=prompt)
    print(analysis_response)

## Introduction to West Bengal Election Results
The West Bengal election results are out, and the **BJP has won a significant majority** of the seats. 

## Key Results
* The BJP has won **206 seats** in the West Bengal assembly elections.
* Mamata Banerjee, the incumbent Chief Minister, has **lost her seat**.
* The Trinamool Congress (TMC) has been **defeated by the BJP**.

## Source Links
For more information, you can visit the following links:
* [NDTV: Election Results 2026](https://www.ndtv.com/elections)
* [Wikipedia: Elections in West Bengal](https://en.wikipedia.org/wiki/Elections_in_West_Bengal)
* [Times of India: West Bengal Election Results 2026](https://timesofindia.indiatimes.com/elections/assembly-elections/west-bengal/results)

## Summary
The **BJP has won the West Bengal election** with a significant majority, marking a major shift in the state's politics.


In [17]:
from IPython.display import Markdown
Markdown(analysis_response)

## Introduction to West Bengal Election Results
The West Bengal election results are out, and the **BJP has won a significant majority** of the seats. 

## Key Results
* The BJP has won **206 seats** in the West Bengal assembly elections.
* Mamata Banerjee, the incumbent Chief Minister, has **lost her seat**.
* The Trinamool Congress (TMC) has been **defeated by the BJP**.

## Source Links
For more information, you can visit the following links:
* [NDTV: Election Results 2026](https://www.ndtv.com/elections)
* [Wikipedia: Elections in West Bengal](https://en.wikipedia.org/wiki/Elections_in_West_Bengal)
* [Times of India: West Bengal Election Results 2026](https://timesofindia.indiatimes.com/elections/assembly-elections/west-bengal/results)

## Summary
The **BJP has won the West Bengal election** with a significant majority, marking a major shift in the state's politics.